# Workflow Runner (Using OpenAI and DuckDB)

If running from within the piglets repository:

```bash
uv sync --extra examples --extra openai --extra duckdb
```

This example runs the currently available workflow stages:

1. Enter the user question.
2. Load a DuckDB search space.
3. Generate a hypothesis with logical planning.
4. Reduce the search space with dual-pathway pruning.
5. Ground the hypothesis in the search space with semantic linking.
6. Verify the grounded search space against table content with parallel data profiling.
7. Finalize the search space with global synthesis.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from piglets import (
    DatabaseConnector,
    DuckDBURL,
    EnterUserQuestion,
    FinalizeSearchSpace,
    GlobalSynthesizer,
    GroundSearchSpace,
    GenerateHypothesis,
    LoadSearchSpace,
    LogicalPlanner,
    DualPathwayPruner,
    ParallelDataProfiler,
    ReduceSearchSpace,
    SemanticLinker,
    VerifySearchSpace,
    WorkflowRunner,
    create_tpch_example_duckdb_db,
)

MODEL_NAME = "gpt-5.2"
DB_PATH = "data/tpch_sf1.duckdb"
QUESTION = """
Which manufacturers saw the largest increase in average revenue per order between 1996 and 1997,
considering only manufacturers with at least 100 orders in both years, and excluding cancelled orders?
"""

create_tpch_example_duckdb_db(db_path=DB_PATH)

database_connector = DatabaseConnector(
    connection=DuckDBURL(database=DB_PATH),
)

runner = WorkflowRunner(
    stages=[
        EnterUserQuestion(QUESTION),
        LoadSearchSpace(database_connector),
        GenerateHypothesis(LogicalPlanner(MODEL_NAME, num_samples=3)),
        ReduceSearchSpace(DualPathwayPruner(MODEL_NAME)),
        GroundSearchSpace(SemanticLinker(MODEL_NAME)),
        VerifySearchSpace(ParallelDataProfiler(MODEL_NAME, database_connector)),
        FinalizeSearchSpace(GlobalSynthesizer(database_connector, MODEL_NAME)),
    ]
)

state = runner.run()


In [ ]:
database_schema = state.search_space.database_schema
profile_result = state.search_space.database_profile_result
synthesis_run = state.search_space.synthesis_run_result

print("Database:", database_schema.name)
print("Database type:", database_schema.database_type)
print("\nFinal search space tables:")
for table_schema in database_schema.table_schemas:
    table_function = (
        table_schema.semantic_annotation.function
        if table_schema.semantic_annotation
        else "Unknown role"
    )
    columns = ", ".join(column_schema.name for column_schema in table_schema.column_schemas)
    print(f"- {table_schema.name}: {table_function}")
    print(f"  Columns: {columns}")

print("\nHypothesis technique:", state.hypothesis.technique)
print("Hypothesis parameters:", state.hypothesis.technique_parameters)
print("\nHypothesis:")
print(state.hypothesis.content)

if profile_result is not None:
    print("\nProfiled tables:", len(profile_result.table_profile_results))
    for table_profile_result in profile_result.table_profile_results:
        print(
            f"- {table_profile_result.table_name}: "
            f"relevant={table_profile_result.relevant}, "
            f"relevant_columns={len(table_profile_result.relevant_columns)}"
        )

if synthesis_run is not None:
    print("\nSynthesis status:", synthesis_run.final_result.status)
    print("Synthesis rounds:", len(synthesis_run.rounds))
    print("Reached round limit:", synthesis_run.reached_limit)
    print("Rejected candidates:", len(synthesis_run.final_result.rejected_candidates))
